# Can a stranger check this receipt

A receipt is only evidence if somebody who was not there can check it. This notebook signs one,
verifies it, then breaks it four different ways.

Before 0.12.0 `sign=True` generated a fresh keypair per call and put the public half **inside** the
receipt it had just signed. Those receipts verify, against a key that came with them, which proves
nothing about who made them. That shape is now refused by name.

In [1]:
import os
import tempfile
from pathlib import Path

# Keep this notebook out of your real key directory.
workspace = Path(tempfile.mkdtemp(prefix="qbc-signing-demo-"))
os.environ["QBC_SIGNING_KEY"] = str(workspace / "signing_key")
os.environ["QBC_TRUSTED_KEYS"] = str(workspace / "trusted_keys")

from qb_compiler.signing import load_or_create_signing_key

key = load_or_create_signing_key()
print("fingerprint:", key.fingerprint)
print("public key :", key.public_b64)
print("file mode  :", oct(os.stat(key.path).st_mode & 0o777))

fingerprint: 5e508bb70048dfd0
public key : jWde991KZaBu4Z1WoFLVYjiqkAhOsap79hWS5Crr/Yo=
file mode  : 0o600


The key is created once and reused. That is the whole point: a reader who has seen one of your
receipts can tell a later one came from the same holder. Publish the public half, keep the file.

In [2]:
from qb_compiler.calibration.static_provider import StaticCalibrationProvider
from qb_compiler.ir.circuit import QBCircuit
from qb_compiler.ir.operations import QBGate
from qb_compiler.passes.mapping import CalibrationMapper, selection_receipt

props = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
).properties

circuit = QBCircuit(n_qubits=3, name="ghz3")
circuit.add_gate(QBGate(name="h", qubits=(0,)))
circuit.add_gate(QBGate(name="cx", qubits=(0, 1)))
circuit.add_gate(QBGate(name="cx", qubits=(1, 2)))

mapper = CalibrationMapper(props)
result = mapper.run(circuit, {})
receipt = selection_receipt(result, calibration=props, sign=True)

print("signing        :", receipt["signing"])
print("key_fingerprint:", receipt["key_fingerprint"])
print("scheme         :", receipt["signature_scheme"])
print("carries the key:", "public_key" in receipt)

signing        : ed25519 (qb-compiler key 5e508bb70048dfd0)
key_fingerprint: 5e508bb70048dfd0
scheme         : ed25519-over-canonical-json-v1
carries the key: False


In [3]:
from qb_compiler.signing import verify_receipt

verdict = verify_receipt(receipt, public_key=key.public_b64)
print(verdict)

VERIFIED: signature verified against key 5e508bb70048dfd0
  schema      : qb.selection_receipt.v1
  key         : 5e508bb70048dfd0
  claims      : which layout was selected, and whether that layout is the one that ran
  claims      : the calibration fingerprint and the measured age of that calibration
  claims      : these exact bytes were signed by the holder of the key named above
  does not say: no claim that the selected layout is optimal
  does not say: the staleness tolerance is a builtin default, not a measurement for that device
  does not say: nothing about whether the numbers inside are correct
  does not say: nothing about whether the run described was worth doing
  does not say: no endorsement by QubitBoost of the party that signed it


The verifier prints what the receipt does not claim as well as what it does. A signature says these
bytes came from that key holder. It says nothing about whether the layout was any good.

## Four ways to fail

Each of these is a separate verdict, and only `VERIFIED` counts as a pass.

In [4]:
import base64
import copy

# 1. Altered after signing.
tampered = copy.deepcopy(receipt)
tampered["selected_layout"] = {"0": 99, "1": 98, "2": 97}
print("tampered   ", verify_receipt(tampered, public_key=key.public_b64).status)

# 2. Signed by somebody else's key.
other = load_or_create_signing_key(workspace / "someone_else")
print("wrong key  ", verify_receipt(receipt, public_key=other.public_b64).status)

# 3. No key supplied at all. This is the one that has to fail closed.
print("no key     ", verify_receipt(receipt).status)

# 4. The pre 0.12.0 shape: a signature travelling with its own key.
legacy = {
    "schema": "qb.selection_receipt.v1",
    "selected_layout": {"0": 1},
    "signature": base64.b64encode(bytes(64)).decode(),
    "signing": "ed25519 (qubitboost_sdk)",
    "public_key": base64.b64encode(bytes(32)).decode(),
}
legacy_verdict = verify_receipt(legacy)
print("legacy     ", legacy_verdict.status)
print()
print(legacy_verdict.reason)

tampered    INVALID_SIGNATURE
wrong key   INVALID_SIGNATURE
no key      NO_KEY
legacy      LEGACY_SELF_SIGNED

receipt embeds the public key its own signature was made with. Signatures made before qb-compiler 0.12.0 used a fresh keypair per receipt and carried it inline, so they attest to nothing. Re-issue the receipt with a persistent key.


## From the command line, which is how an auditor will do it

Publish the public key next to the receipt. Neither is a secret.

In [5]:
import json
import subprocess

from qb_compiler.signing import export_public_key

receipt_path = workspace / "receipt.json"
receipt_path.write_text(json.dumps(receipt, indent=2))
key_path = export_public_key(workspace / "qbc-public-key.txt", key=key)

for args in (
    ["qbc", "verify-receipt", str(receipt_path), "--key", str(key_path)],
    ["qbc", "verify-receipt", str(receipt_path)],
):
    done = subprocess.run(args, capture_output=True, text=True)
    print("$ qbc verify-receipt ...", "--key" if "--key" in args else "(no key)")
    print(done.stdout.strip())
    print("exit code:", done.returncode)
    print()

$ qbc verify-receipt ... --key
VERIFIED: signature verified against key 5e508bb70048dfd0
  schema      : qb.selection_receipt.v1
  key         : 5e508bb70048dfd0
  claims      : which layout was selected, and whether that layout is the one that ran
  claims      : the calibration fingerprint and the measured age of that calibration
  claims      : these exact bytes were signed by the holder of the key named above
  does not say: no claim that the selected layout is optimal
  does not say: the staleness tolerance is a builtin default, not a measurement for that device
  does not say: nothing about whether the numbers inside are correct
  does not say: nothing about whether the run described was worth doing
  does not say: no endorsement by QubitBoost of the party that signed it
exit code: 0



$ qbc verify-receipt ... (no key)
NO_KEY: receipt is signed but no public key was supplied, so the signature cannot be checked. Pass the signer's public key, or add it to the trusted-keys file.
  schema      : qb.selection_receipt.v1
  key         : 5e508bb70048dfd0
  does not say: no claim that the selected layout is optimal
  does not say: the staleness tolerance is a builtin default, not a measurement for that device
  does not say: nothing about whether the numbers inside are correct
  does not say: nothing about whether the run described was worth doing
  does not say: no endorsement by QubitBoost of the party that signed it
exit code: 1



Exit 0 verified, 1 could not be checked, 2 does not verify. That is what a CI job reads.

Verification needs no account, no network, and no compiled crypto library: it falls back to a pure
Python Ed25519 implementation, so `pip install qb-compiler` is enough to check anybody's receipt.
That asymmetry is deliberate. Issuing receipts at scale is a product. Checking one is a right.

In [6]:
import shutil

shutil.rmtree(workspace)
print("temporary keys removed")

temporary keys removed
